# NYC Airbnb Market Explorer – Business Analysis

Answers key business questions using the cleaned listings dataset.

In [ ]:
import pandas as pd
df = pd.read_csv("../data/processdata/listings_cleaned.csv")
print("Loaded shape:", df.shape)

## Market Area

### 1. Neighbourhoods with the most listings

In [ ]:
print(df["neighbourhood_cleansed"].value_counts().head(12))

### 2. Room type distribution

In [ ]:
print(df["room_type"].value_counts())
print((df["room_type"].value_counts(normalize=True)*100).round(2))

### 3. Geographic distribution (borough level)

In [ ]:
print(df["neighbourhood_group_cleansed"].value_counts())

## Pricing Area

### 4. Average price by neighbourhood and room type (sample)

In [ ]:
avg_price = (
    df.groupby(["neighbourhood_cleansed", "room_type"])["price"]
    .mean()
    .round(2)
    .sort_values(ascending=False)
)
print(avg_price.head(15))

### 5–6. Expensive areas and extreme prices

In [ ]:
print("Overall average price:", round(df["price"].mean(), 2))
print("\nHighest prices:")
print(df[["neighbourhood_cleansed", "price"]].sort_values("price", ascending=False).head(10))
print("\nLowest prices:")
print(df[["neighbourhood_cleansed", "price"]].sort_values("price").head(10))

## Hosts Area

### 7. Hosts managing the most listings

In [ ]:
host_counts = df.groupby(["host_id", "host_name"]).size().sort_values(ascending=False)
print(host_counts.head(12))

### 8. Multi-listing hosts prevalence

In [ ]:
multi = (host_counts >= 2).sum()
single = (host_counts == 1).sum()
print("Multi-listing hosts:", multi)
print("Single-listing hosts:", single)
print("Share of hosts with 2+ listings: {:.1f}%".format(100 * multi / (multi + single)))

## Performance Area

### 10. Top reviewed listings

In [ ]:
print(df[["id", "host_name", "neighbourhood_cleansed", "number_of_reviews"]]
      .sort_values("number_of_reviews", ascending=False).head(12))

### 11. Higher rated neighbourhoods (min 15 listings)

In [ ]:
rating_by_nb = (
    df.groupby("neighbourhood_cleansed")
    .agg(avg_rating=("review_scores_rating", "mean"), n=("id", "count"))
    .query("n >= 15")
    .sort_values("avg_rating", ascending=False)
)
print(rating_by_nb.head(12).round(2))

### 12. Price vs rating vs reviews summary

In [ ]:
print("Average price:", round(df["price"].mean(), 2))
print("Average rating:", round(df["review_scores_rating"].mean(), 2))
print("Average reviews:", round(df["number_of_reviews"].mean(), 2))

## Availability & Opportunity

### 13–14. Occupancy patterns

In [ ]:
occ = df.groupby(["neighbourhood_cleansed", "room_type"])["estimated_occupancy_l365d"].mean()
print("Highest average occupancy:")
print(occ.sort_values(ascending=False).head(10))
print("\nOverall average occupancy:", round(df["estimated_occupancy_l365d"].mean(), 2))

### 15. Opportunity candidates for a new host

In [ ]:
opp = (
    df.groupby(["neighbourhood_cleansed", "room_type"])
    .agg(
        avg_price=("price", "mean"),
        avg_rating=("review_scores_rating", "mean"),
        avg_reviews=("number_of_reviews", "mean"),
        n=("id", "count")
    )
    .query("n >= 12 and 50 <= avg_price <= 400")
    .sort_values(["avg_rating", "avg_reviews"], ascending=False)
)
print(opp.head(12).round(2))